# 申鹤 RVC 语音
Runtime → Run all → 拿到公网 URL

In [ ]:
# 1. 安装依赖
!pip install -q torch torchaudio fairseq librosa soundfile pyworld numpy scipy
!pip install -q edge_tts gradio huggingface_hub
!apt-get update -qq 2>/dev/null && apt-get install -y -qq ffmpeg 2>/dev/null

import torch, edge_tts, gradio
print(f'PyTorch {torch.__version__} | GPU: {torch.cuda.is_available()}')
print('✅ 依赖就绪')

In [ ]:
# 2. 克隆 RVC + 下载模型
import sys, os

if not os.path.exists('/content/rvc'):
    !git clone --depth 1 https://github.com/RVC-Project/Retrieval-based-Voice-Conversion-WebUI.git /content/rvc -q

sys.path.insert(0, '/content/rvc')
sys.path.insert(0, '/content/rvc/infer/lib')
sys.path.insert(0, '/content/rvc/infer/modules')
os.chdir('/content/rvc')

MODEL_DIR = '/content/models'
os.makedirs(MODEL_DIR, exist_ok=True)
os.environ['weight_root'] = MODEL_DIR
os.environ['index_root'] = MODEL_DIR

from huggingface_hub import hf_hub_download

print('下载 HuBERT...')
hf_hub_download('lj1995/VoiceConversionWebUI', filename='hubert_base.pt',
                local_dir='/content/rvc/assets/hubert')

print('下载申鹤模型...')
for f in ['genshin_impact/shenhe_e250_s11250.pth', 'genshin_impact/shenhe_pitch.index']:
    hf_hub_download('Cinnamomo/rvc_models', filename=f, local_dir=MODEL_DIR)

INDEX = f'{MODEL_DIR}/genshin_impact/shenhe_pitch.index'
print('✅ 模型就绪')

In [ ]:
# 3. 加载模型
import torch
from infer.modules.vc.modules import VC

class Cfg:
    is_half = True
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    n_cpu = 2; x_pad = 3; x_query = 10; x_center = 60; x_max = 65
    f0method = 'harvest'

vc = VC(Cfg())
vc.get_vc('shenhe_e250_s11250.pth')
print(f'✅ 申鹤模型加载完毕 | {Cfg.device}')

In [ ]:
# 4. 语音生成函数
import asyncio, uuid, numpy as np
import soundfile as sf

def shenhe_speak(text):
    if not text.strip():
        return None
    # TTS
    mp3 = f'/tmp/tts_{uuid.uuid4().hex[:8]}.mp3'
    async def gen():
        await edge_tts.Communicate(text, 'zh-CN-XiaoxiaoNeural', rate='-10%').save(mp3)
    asyncio.run(gen())
    # RVC
    r = vc.vc_single(0, mp3, 0, None, 'harvest', INDEX, '', 0.5, 3, 0, 0.25, 0.33)
    info, (sr, audio) = r
    if audio is None:
        return None
    out = f'/tmp/shenhe_{uuid.uuid4().hex[:8]}.wav'
    arr = audio.cpu().numpy() if hasattr(audio, 'cpu') else audio
    sf.write(out, arr.reshape(-1), sr)
    return out

# 测试
result = shenhe_speak('我名申鹤，命格孤煞。')
print(f'✅ 测试通过: {result}' if result else '❌ 测试失败')

In [ ]:
# 5. 启动 Gradio 公网 API
import gradio as gr

def api_tts(text):
    path = shenhe_speak(text)
    if path:
        return path
    return None

demo = gr.Interface(
    fn=api_tts,
    inputs=gr.Textbox(label='输入中文文本'),
    outputs=gr.Audio(label='申鹤语音'),
    title='申鹤 RVC 语音 API',
    description='输入中文 → 输出申鹤(CV秦紫翼)声线语音'
)

demo.queue().launch(share=True)